In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from datetime import datetime
from pathlib import Path
import tarfile

import torch
import pandas as pd

import pytorch_lightning as pl
from pytorch_lightning.loggers import CSVLogger

from src.dataset import ASRDataset
from src.normalizers.alphabet_normalizer import RussianAlphabetWordNormalizer
from src.normalizers.unspittable_normalizer import UnspittableNormalizer
from src.augmenter import Augmenter
from pytorch_lightning.callbacks import ModelCheckpoint

torch.set_float32_matmul_precision("high")

# Configuration

In [3]:
SAMPLE_RATE = 16000
N_MELS = 80
N_FFT = 256
HOP_LENGTH = 160

BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3

dataset_path = base_dir = Path("data")
train_dataset_dir = (base_dir / "train") 
dev_dataset_dir =(base_dir / "dev")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Loading data

In [4]:
train_dataset_dir.mkdir(parents=True, exist_ok=True)
dev_dataset_dir.mkdir(parents=True, exist_ok=True)

train_csv = base_dir / "train.csv"
dev_csv = base_dir / "dev.csv"

# Extract datasets if not already extracted
if not train_csv.is_file():
    with tarfile.open(train_dataset_dir / "train.tar.gz", "r:gz") as tar:
        tar.extractall(dataset_path)

if not dev_csv.is_file():
    with tarfile.open(dev_dataset_dir / "dev.tar.gz", "r:gz") as tar:
        tar.extractall(dataset_path)

In [5]:
from src.normalizers.unspittable_normalizer import UnspittableNormalizer
#normalizer = RussianAlphabetWordNormalizer()

normalizer = UnspittableNormalizer()
vocab, vocab_size = normalizer.get_vocab()

print(vocab)

[0, 1000, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200, 300, 400, 500, 600, 700, 800, 900]


In [6]:
print(vocab_size)

38


In [7]:
NOISE_DIR = Path("data/musan/noise")

In [8]:
NOISE_DIR.exists()

True

In [33]:
import os

print("CWD:", os.getcwd())

CWD: /mnt/c/Users/Григорий/ai-talent-hub-itmo-speech-course/group-projects/gp1


In [9]:
augmenter = Augmenter(
    sample_rate=SAMPLE_RATE,
    noise_dir=NOISE_DIR,
    noise_prob=0.5,
    speed_prob=0.3,
    gain_prob=0.4,
    shift_prob=0.5,
    clip_prob=0.2,
    reverb_prob=0.3,
)


train_dataset = ASRDataset(
    csv_path=train_csv,
    root_dir=dataset_path,
    augmenter=augmenter,
    spec_augment = True,
    normalizer=normalizer,
    sample_rate=SAMPLE_RATE,
    n_mels=N_MELS,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    device=device,
)

dev_dataset = ASRDataset(
    csv_path=dev_csv,
    root_dir=dataset_path,
    augmenter=None,
    normalizer=normalizer,
    sample_rate=SAMPLE_RATE,
    n_mels=N_MELS,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    device=device,
)

/home/gorlov/asr_env/lib/python3.12/site-packages/torchaudio/functional/functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (80) may be set too high. Or, the value for `n_freqs` (129) may be set too low.
  warnings.warn(
Generating label files: 100%|██████████| 12553/12553 [00:00<00:00, 212641.50it/s]
/home/gorlov/asr_env/lib/python3.12/site-packages/torchaudio/functional/functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (80) may be set too high. Or, the value for `n_freqs` (129) may be set too low.
  warnings.warn(
Generating label files: 100%|██████████| 2265/2265 [00:00<00:00, 190501.08it/s]


## Final DataLoaders

In [10]:
def collate_fn(batch):
    mels, labels, spk_ids = zip(*batch)
    mel_lenghts = torch.tensor([mel.shape[0] for mel in mels])
    mels = torch.nn.utils.rnn.pad_sequence(mels, batch_first=True)

    labels_lengths = torch.tensor([len(label) for label in labels])
    # labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True)
    labels = torch.cat(labels)
    return mels, labels, mel_lenghts, labels_lengths, list(spk_ids)

In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    persistent_workers=True,
    pin_memory=True
)

## Check dataset view (Optional)

In [8]:
# test_dataset = ASRDataset(
#     csv_path=base_dir / "test.csv",
#     root_dir=dataset_path,
#     augmenter=Augmenter(),
#     normalizer=normalizer,
#     sample_rate=SAMPLE_RATE,
#     n_mels=N_MELS,
#     n_fft=N_FFT,
#     hop_length=HOP_LENGTH,
#     device=device,
# )

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     collate_fn=collate_fn,
#     num_workers=0
# )

# for x, texts, lengths, t_lengths in test_loader:
#     print(f"{x.shape=}, {texts=}, {lengths=}")

# Model training

In [12]:
train_df = pd.read_csv(train_csv)
t_spk_ids = train_df["spk_id"].tolist()

dev_df = pd.read_csv(dev_csv)
d_spk_ids = dev_df["spk_id"].tolist()

in_domain_speakers = set(t_spk_ids) & set(d_spk_ids)
out_of_domain_speakers = set(d_spk_ids) - set(t_spk_ids)

In [13]:
EXPERIMENT = "unsplittable-v2"

In [ ]:
from src.models.simple_cnn import ASRModel
from src.trainer import LightningASR

model = ASRModel(vocab_size)
lit_model = LightningASR(
    model, vocab, normalizer=normalizer,
    in_domain_speakers=in_domain_speakers, lr=LR)


ckpt_cb = ModelCheckpoint(
    dirpath="logs/unspittable_v2/checkpoints",
    filename="{epoch:02d}-{val/harmonic_cer:.4f}",
    monitor="val/harmonic_cer",
    mode="min",
    save_top_k=2,
    save_last=True,
)

logger = CSVLogger(
    "logs", name=EXPERIMENT
)
logger.log_hyperparams(
    {
        "sample_rate": SAMPLE_RATE,
        "n_mels": N_MELS,
        "n_fft": N_FFT,
        "hop_lenght": HOP_LENGTH,
        "batch": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "normalizer": str(normalizer),
        "augmentation": augmenter.get_params(),
    }
)

In [15]:
trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    devices=1,
    log_every_n_steps=100,
    logger=logger,
    callbacks=[ckpt_cb],
)

trainer.fit(lit_model, train_loader, val_dataloaders=dev_loader)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name     ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model    │ ASRModel │  4.8 M │ train │     0 │
│ 1 │ ctc_loss │ CTCLoss  │      0 │ train │     0 │
└───┴──────────┴──────────┴────────┴───────┴───────┘

Trainable params: 4.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.8 M                                                                                                
Total estimated model params size (MB): 19                                                                         
Modules in train mode: 9                                                                                           
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/home/gorlov/asr_env/lib/python3.12/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, 
LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

`Trainer.fit` stopped: `max_epochs=20` reached.


# Evaluate results

In [16]:
from src.metrics import evaluate_by_speaker

## Baseline Results

In [ ]:
results = evaluate_by_speaker( #baseline
    lit_model.model,
    dev_loader,
    normalizer,
    d_spk_ids,
    in_domain_speakers
)

print("In-domain CER:", results["in_domain"])
print("Out-of-domain CER:", results["out_of_domain"])

score = ((2 * results["in_domain"] * results["out_of_domain"]) / (results["in_domain"] + results["out_of_domain"])) * 100
print(f"Harmonic CER mean: {score}%")

print("\nPer speaker:")
for spk, val in results["per_speaker"].items():
    print(spk, f"{val:.4f}")


In-domain CER: 0.20063888888888887
Out-of-domain CER: 0.21885885885885883
Harmonic CER mean: 20.9353201540126%

Per speaker:
spk_J 0.2192
spk_I 0.2768
spk_K 0.2265
spk_H 0.1709
spk_F 0.2590
spk_C 0.2017
spk_D 0.2117
spk_B 0.1512
spk_E 0.1210
spk_A 0.2593


In [16]:
results = evaluate_by_speaker(
    lit_model.model,
    train_loader,
    normalizer,
    t_spk_ids,
    t_spk_ids
)

print("In-domain CER:", results["in_domain"])
print("Out-of-domain CER:", results["out_of_domain"])

print("\nPer speaker:")
for spk, val in results["per_speaker"].items():
    print(spk, f"{val:.4f}")

In-domain CER: 0.011529780397780078
Out-of-domain CER: None

Per speaker:
spk_E 0.0114
spk_B 0.0117
spk_A 0.0117
spk_C 0.0134
spk_D 0.0112
spk_F 0.0095


## Spec-augment results

In [12]:
results = evaluate_by_speaker( #baseline
    lit_model.model,
    dev_loader,
    normalizer,
    d_spk_ids,
    in_domain_speakers
)

print("In-domain CER:", results["in_domain"])
print("Out-of-domain CER:", results["out_of_domain"])

score = ((2 * results["in_domain"] * results["out_of_domain"]) / (results["in_domain"] + results["out_of_domain"])) * 100
print(f"Harmonic CER mean: {score}%")

print("\nPer speaker:")
for spk, val in results["per_speaker"].items():
    print(spk, f"{val:.4f}")


In-domain CER: 0.10747222222222222
Out-of-domain CER: 0.13974974974974974
Harmonic CER mean: 12.150389417901582%

Per speaker:
spk_J 0.1594
spk_I 0.1778
spk_K 0.1628
spk_H 0.0923
spk_F 0.1603
spk_C 0.1043
spk_D 0.1550
spk_B 0.0375
spk_E 0.0703
spk_A 0.1173


In [13]:
results = evaluate_by_speaker(
    lit_model.model,
    train_loader,
    normalizer,
    t_spk_ids,
    t_spk_ids
)

print("In-domain CER:", results["in_domain"])
print("Out-of-domain CER:", results["out_of_domain"])

print("\nPer speaker:")
for spk, val in results["per_speaker"].items():
    print(spk, f"{val:.4f}")

In-domain CER: 0.04390716694548447
Out-of-domain CER: None

Per speaker:
spk_E 0.0439
spk_B 0.0420
spk_A 0.0409
spk_C 0.0493
spk_D 0.0438
spk_F 0.0473


## Real-Noise added Results

In [15]:
results = evaluate_by_speaker( #baseline
    lit_model.model,
    dev_loader,
    normalizer,
    d_spk_ids,
    in_domain_speakers
)

print("In-domain CER:", results["in_domain"])
print("Out-of-domain CER:", results["out_of_domain"])

score = ((2 * results["in_domain"] * results["out_of_domain"]) / (results["in_domain"] + results["out_of_domain"])) * 100
print(f"Harmonic CER mean: {score}%")

print("\nPer speaker:")
for spk, val in results["per_speaker"].items():
    print(spk, f"{val:.4f}")


In-domain CER: 0.04413888888888889
Out-of-domain CER: 0.10781781781781781
Harmonic CER mean: 6.263571755458537%

Per speaker:
spk_J 0.0869
spk_I 0.2046
spk_K 0.1482
spk_H 0.0310
spk_F 0.0697
spk_C 0.0460
spk_D 0.0525
spk_B 0.0317
spk_E 0.0133
spk_A 0.0517


In [16]:
results = evaluate_by_speaker(
    lit_model.model,
    train_loader,
    normalizer,
    t_spk_ids,
    t_spk_ids
)

print("In-domain CER:", results["in_domain"])
print("Out-of-domain CER:", results["out_of_domain"])

print("\nPer speaker:")
for spk, val in results["per_speaker"].items():
    print(spk, f"{val:.4f}")

In-domain CER: 0.03804269895642476
Out-of-domain CER: None

Per speaker:
spk_E 0.0414
spk_B 0.0349
spk_A 0.0342
spk_C 0.0325
spk_D 0.0381
spk_F 0.0379


## Real-Noise + Unsplittable Normalizer 

In [17]:
results = evaluate_by_speaker( #baseline
    lit_model.model,
    dev_loader,
    normalizer,
    d_spk_ids,
    in_domain_speakers
)

print("In-domain CER:", results["in_domain"])
print("Out-of-domain CER:", results["out_of_domain"])

score = ((2 * results["in_domain"] * results["out_of_domain"]) / (results["in_domain"] + results["out_of_domain"])) * 100
print(f"Harmonic CER mean: {score}%")

print("\nPer speaker:")
for spk, val in results["per_speaker"].items():
    print(spk, f"{val:.4f}")


In-domain CER: 0.03836111111111111
Out-of-domain CER: 0.06545545545545546
Harmonic CER mean: 4.8372703559698405%

Per speaker:
spk_J 0.0739
spk_I 0.0891
spk_K 0.0953
spk_H 0.0333
spk_F 0.0667
spk_C 0.0460
spk_D 0.0408
spk_B 0.0220
spk_E 0.0120
spk_A 0.0427


In [18]:
results = evaluate_by_speaker(
    lit_model.model,
    train_loader,
    normalizer,
    t_spk_ids,
    t_spk_ids
)

print("In-domain CER:", results["in_domain"])
print("Out-of-domain CER:", results["out_of_domain"])

print("\nPer speaker:")
for spk, val in results["per_speaker"].items():
    print(spk, f"{val:.4f}")

In-domain CER: 0.04076050877612257
Out-of-domain CER: None

Per speaker:
spk_E 0.0396
spk_B 0.0437
spk_A 0.0387
spk_C 0.0444
spk_D 0.0355
spk_F 0.0421


In [26]:
from pathlib import Path
for p in Path("logs").rglob("metrics.csv"):
    print(p)

logs/realnoise/version_0/version_0/metrics.csv
logs/specaugment/version_0/version_0/metrics.csv
logs/trainer-fix/version_0/version_0/metrics.csv
logs/unspittable/version_0/Real-noise_unsplittableNormalizer/version_0/metrics.csv


In [27]:
import pandas as pd
from pathlib import Path

for metrics in sorted(Path("logs").rglob("metrics.csv")):
    df = pd.read_csv(metrics)
    if "val/harmonic_cer" not in df.columns:
        print(f"{metrics}: no val/harmonic_cer column — skipping")
        continue
    vc = df["val/harmonic_cer"].dropna()
    exp = metrics.parent.relative_to("logs")
    print(f"{str(exp):60s}  last={vc.iloc[-1]:.4f}  best={vc.min():.4f}  @row={vc.idxmin()}")


realnoise/version_0/version_0                                 last=0.0626  best=0.0603  @row=58
specaugment/version_0/version_0                               last=0.1215  best=0.1159  @row=82
trainer-fix/version_0/version_0                               last=0.2094  best=0.1996  @row=92
unspittable/version_0/Real-noise_unsplittableNormalizer/version_0  last=0.0457  best=0.0399  @row=92


In [22]:
df = pd.read_csv("logs/unsplittable-v2/version_0/metrics.csv")
epoch_best = df.loc[df["val/harmonic_cer"].idxmin(), ["epoch", "val/harmonic_cer"]]
print(epoch_best)

for e in [13, 17]:
    val = df.loc[df["epoch"] == e, "val/harmonic_cer"].dropna().iloc[-1]
    print(f"Epoch {e}: val/harmonic_cer = {val:.4f}")


epoch               17.000000
val/harmonic_cer     0.032351
Name: 87, dtype: float64
Epoch 13: val/harmonic_cer = 0.0373
Epoch 17: val/harmonic_cer = 0.0324
